In [1]:
from pyscf import gto, scf, lib
import numpy as np
from pyscf.hessian import rhf as rhf_hess
from pyscf.df.hessian import rhf as df_rhf_hess
from functools import partial
import scipy
from pyscf.df.grad.rhf import _int3c_wrapper

lib.num_threads(16)
np.set_printoptions(5, suppress=True, linewidth=150)
np.einsum = partial(np.einsum, optimize="greedy")

In [2]:
xyz = """
N  0   0   0
H  1.0 0.1 0.2
H  0.3 1.1 0.2
H  0.1 0.1 1.2
"""

mol = gto.Mole(atom=xyz, basis="def2-TZVP", max_memory=32000).build()

In [3]:
mf = scf.RHF(mol).density_fit()
mf.with_df.build()
mf.run()

converged SCF energy = -56.1132387662624


## 参考值计算

计算 RI-JK RHF Hessian 的参考值 `de_ref`。

该分解的目标是：

$$\text{de\_ref} = \text{de\_hcore} + \text{de\_ovlp} + \text{de\_J\_(basis\_2nd)} + \text{de\_J\_(basis\_1st\_aux\_1st)} + \text{de\_J\_(aux\_2nd)} - \text{de\_K\_(basis\_2nd)} - \text{de\_K\_(basis\_1st\_aux\_1st)} - \text{de\_K\_(aux\_2nd)} + \text{de\_cphf} + \text{de\_nuc}$$

其中各贡献按 PySCF 的 notation 分类：
- **basis\_2nd**: 全部导数在轨道基上 (0 在辅助基)，包括 $(20|0)(0|00)$, $(11|0)(0|00)$, $(10|0)(0|10)$
- **basis\_1st\_aux\_1st**: 轨道基 1阶 + 辅助基 1阶，包括 $(10|1)(0|00)$, $(10|0)(0|1)(0|00)$, $(10|0)(1|0)(0|00)$, $(10|0)(1|00)$ 等
- **aux\_2nd**: 全部导数在辅助基上 (0 在轨道基)，包括 $(00|2)(0|00)$, $(00|0)(1|1)(0|00)$ 等

In [4]:
mf_hess = mf.Hessian().run()
de_ref = mf_hess.de.copy()
print("de_ref shape:", de_ref.shape)

tmp_contrib1, fp: -40.4820033252071
tmp_contrib2, fp: 40.49762230517417
tmp_contrib3, fp: -4.423643843762165
tmp_contrib4, fp: -20362.502758972907
tmp_contrib5, fp: 10181.02765040996
tmp_contrib6, fp: 10181.475120771258
tmp_contrib7, fp: 100.26786686092383
tmp_contrib8, fp: -100.33444387990247
tmp_contrib1, fp: -40.4820033252071
[[[[-218.0347     0.01337    0.57194]
   [   0.01337 -219.39356    0.244  ]
   [   0.57194    0.244   -219.68089]]

  [[   0.         0.         0.     ]
   [   0.         0.         0.     ]
   [   0.         0.         0.     ]]

  [[   0.         0.         0.     ]
   [   0.         0.         0.     ]
   [   0.         0.         0.     ]]

  [[   0.         0.         0.     ]
   [   0.         0.         0.     ]
   [   0.         0.         0.     ]]]


 [[[   0.         0.         0.     ]
   [   0.         0.         0.     ]
   [   0.         0.         0.     ]]

  [[   0.0943     0.06305    0.18068]
   [   0.06305   -1.07026    0.02518]
   [   0.18

## 基本量提取

In [5]:
mo_coeff = mf.mo_coeff
mo_occ = mf.mo_occ
mo_energy = mf.mo_energy
nao, nmo = mo_coeff.shape
occ_coeff = mo_coeff[:, mo_occ > 0]
occ_occupation = mo_occ[mo_occ > 0]
nocc = occ_coeff.shape[1]
dm0 = np.dot(occ_coeff, occ_coeff.T) * 2
dme0 = np.einsum('pi,qi,i->pq', occ_coeff, occ_coeff, mo_energy[mo_occ > 0]) * 2
natm = mol.natm
atmlst = range(natm)
aoslices = mol.aoslice_by_atom()
aux = mf.with_df.auxmol
auxslices = aux.aoslice_by_atom()
naux = aux.nao

# 基本分解

## 核排斥贡献

In [6]:
de_nuc = rhf_hess.hess_nuc(mol)

## `_partial_hess_ejk` 分解

利用 `auxbasis_response` 参数的不同取值来提取不同阶数的辅助基贡献：

- `auxbasis_response = 0`: 只计算轨道基导数 → **basis\_2nd** 贡献
- `auxbasis_response = 1`: 添加一阶辅助基响应 (J factor 1.0, K factor 0.5)
- `auxbasis_response = 2`: 添加完整一阶 + 二阶辅助基响应 (J factor 2.0, K factor 1.0)

分解公式：
- `J_basis_2nd = ej_aux0` (aux0 结果)
- `J_basis_1st_aux_1st = 2 × (ej_aux1 - ej_aux0)` (修正为 full factor 2.0)
- `J_aux_2nd = ej_aux2 - 2×ej_aux1 + ej_aux0`
- K 同理

In [7]:
# auxbasis_response = 0: only orbital derivatives (basis_2nd)
hessobj_aux0 = mf.Hessian()
hessobj_aux0.auxbasis_response = 0
e1_aux0, ej_aux0, ek_aux0 = df_rhf_hess._partial_hess_ejk(hessobj_aux0)

tmp_contrib1, fp: 0.0
tmp_contrib2, fp: 0.0
tmp_contrib3, fp: 0.0
tmp_contrib4, fp: 0.0
tmp_contrib5, fp: 0.0
tmp_contrib6, fp: 0.0
tmp_contrib7, fp: 0.0
tmp_contrib8, fp: 0.0
tmp_contrib1, fp: 0.0
[[[[0. 0. 0.]
   [0. 0. 0.]
   [0. 0. 0.]]

  [[0. 0. 0.]
   [0. 0. 0.]
   [0. 0. 0.]]

  [[0. 0. 0.]
   [0. 0. 0.]
   [0. 0. 0.]]

  [[0. 0. 0.]
   [0. 0. 0.]
   [0. 0. 0.]]]


 [[[0. 0. 0.]
   [0. 0. 0.]
   [0. 0. 0.]]

  [[0. 0. 0.]
   [0. 0. 0.]
   [0. 0. 0.]]

  [[0. 0. 0.]
   [0. 0. 0.]
   [0. 0. 0.]]

  [[0. 0. 0.]
   [0. 0. 0.]
   [0. 0. 0.]]]


 [[[0. 0. 0.]
   [0. 0. 0.]
   [0. 0. 0.]]

  [[0. 0. 0.]
   [0. 0. 0.]
   [0. 0. 0.]]

  [[0. 0. 0.]
   [0. 0. 0.]
   [0. 0. 0.]]

  [[0. 0. 0.]
   [0. 0. 0.]
   [0. 0. 0.]]]


 [[[0. 0. 0.]
   [0. 0. 0.]
   [0. 0. 0.]]

  [[0. 0. 0.]
   [0. 0. 0.]
   [0. 0. 0.]]

  [[0. 0. 0.]
   [0. 0. 0.]
   [0. 0. 0.]]

  [[0. 0. 0.]
   [0. 0. 0.]
   [0. 0. 0.]]]]
tmp_contrib2, fp: 0.0
[[[[0. 0. 0.]
   [0. 0. 0.]
   [0. 0. 0.]]

  [[0. 0. 0.]
   [0. 0. 0

In [8]:
# auxbasis_response = 1: 1st-order aux response (J factor 1.0, K factor 0.5)
hessobj_aux1 = mf.Hessian()
hessobj_aux1.auxbasis_response = 1
e1_aux1, ej_aux1, ek_aux1 = df_rhf_hess._partial_hess_ejk(hessobj_aux1)

tmp_contrib1, fp: 0.0
tmp_contrib2, fp: 0.0
tmp_contrib3, fp: 0.0
tmp_contrib4, fp: 0.0
tmp_contrib5, fp: 0.0
tmp_contrib6, fp: 0.0
tmp_contrib7, fp: 0.0
tmp_contrib8, fp: 0.0
tmp_contrib1, fp: 0.0
[[[[0. 0. 0.]
   [0. 0. 0.]
   [0. 0. 0.]]

  [[0. 0. 0.]
   [0. 0. 0.]
   [0. 0. 0.]]

  [[0. 0. 0.]
   [0. 0. 0.]
   [0. 0. 0.]]

  [[0. 0. 0.]
   [0. 0. 0.]
   [0. 0. 0.]]]


 [[[0. 0. 0.]
   [0. 0. 0.]
   [0. 0. 0.]]

  [[0. 0. 0.]
   [0. 0. 0.]
   [0. 0. 0.]]

  [[0. 0. 0.]
   [0. 0. 0.]
   [0. 0. 0.]]

  [[0. 0. 0.]
   [0. 0. 0.]
   [0. 0. 0.]]]


 [[[0. 0. 0.]
   [0. 0. 0.]
   [0. 0. 0.]]

  [[0. 0. 0.]
   [0. 0. 0.]
   [0. 0. 0.]]

  [[0. 0. 0.]
   [0. 0. 0.]
   [0. 0. 0.]]

  [[0. 0. 0.]
   [0. 0. 0.]
   [0. 0. 0.]]]


 [[[0. 0. 0.]
   [0. 0. 0.]
   [0. 0. 0.]]

  [[0. 0. 0.]
   [0. 0. 0.]
   [0. 0. 0.]]

  [[0. 0. 0.]
   [0. 0. 0.]
   [0. 0. 0.]]

  [[0. 0. 0.]
   [0. 0. 0.]
   [0. 0. 0.]]]]
tmp_contrib2, fp: 0.0
[[[[0. 0. 0.]
   [0. 0. 0.]
   [0. 0. 0.]]

  [[0. 0. 0.]
   [0. 0. 0

In [9]:
# auxbasis_response = 2: full aux response (default, J factor 2.0, K factor 1.0)
hessobj_aux2 = mf.Hessian()
hessobj_aux2.auxbasis_response = 2
e1_aux2, ej_aux2, ek_aux2 = df_rhf_hess._partial_hess_ejk(hessobj_aux2)

tmp_contrib1, fp: -40.4820033252071
tmp_contrib2, fp: 40.49762230517417
tmp_contrib3, fp: -4.423643843762165
tmp_contrib4, fp: -20362.502758972907
tmp_contrib5, fp: 10181.02765040996
tmp_contrib6, fp: 10181.475120771258
tmp_contrib7, fp: 100.26786686092383
tmp_contrib8, fp: -100.33444387990247
tmp_contrib1, fp: -40.4820033252071
[[[[-218.0347     0.01337    0.57194]
   [   0.01337 -219.39356    0.244  ]
   [   0.57194    0.244   -219.68089]]

  [[   0.         0.         0.     ]
   [   0.         0.         0.     ]
   [   0.         0.         0.     ]]

  [[   0.         0.         0.     ]
   [   0.         0.         0.     ]
   [   0.         0.         0.     ]]

  [[   0.         0.         0.     ]
   [   0.         0.         0.     ]
   [   0.         0.         0.     ]]]


 [[[   0.         0.         0.     ]
   [   0.         0.         0.     ]
   [   0.         0.         0.     ]]

  [[   0.0943     0.06305    0.18068]
   [   0.06305   -1.07026    0.02518]
   [   0.18

In [10]:
hessobj_tmp = mf.Hessian()
hessobj_tmp.auxbasis_response = 0
hessobj_tmp.run()
hessobj_tmp.de

tmp_contrib1, fp: 0.0
tmp_contrib2, fp: 0.0
tmp_contrib3, fp: 0.0
tmp_contrib4, fp: 0.0
tmp_contrib5, fp: 0.0
tmp_contrib6, fp: 0.0
tmp_contrib7, fp: 0.0
tmp_contrib8, fp: 0.0
tmp_contrib1, fp: 0.0
[[[[0. 0. 0.]
   [0. 0. 0.]
   [0. 0. 0.]]

  [[0. 0. 0.]
   [0. 0. 0.]
   [0. 0. 0.]]

  [[0. 0. 0.]
   [0. 0. 0.]
   [0. 0. 0.]]

  [[0. 0. 0.]
   [0. 0. 0.]
   [0. 0. 0.]]]


 [[[0. 0. 0.]
   [0. 0. 0.]
   [0. 0. 0.]]

  [[0. 0. 0.]
   [0. 0. 0.]
   [0. 0. 0.]]

  [[0. 0. 0.]
   [0. 0. 0.]
   [0. 0. 0.]]

  [[0. 0. 0.]
   [0. 0. 0.]
   [0. 0. 0.]]]


 [[[0. 0. 0.]
   [0. 0. 0.]
   [0. 0. 0.]]

  [[0. 0. 0.]
   [0. 0. 0.]
   [0. 0. 0.]]

  [[0. 0. 0.]
   [0. 0. 0.]
   [0. 0. 0.]]

  [[0. 0. 0.]
   [0. 0. 0.]
   [0. 0. 0.]]]


 [[[0. 0. 0.]
   [0. 0. 0.]
   [0. 0. 0.]]

  [[0. 0. 0.]
   [0. 0. 0.]
   [0. 0. 0.]]

  [[0. 0. 0.]
   [0. 0. 0.]
   [0. 0. 0.]]

  [[0. 0. 0.]
   [0. 0. 0.]
   [0. 0. 0.]]]]
tmp_contrib2, fp: 0.0
[[[[0. 0. 0.]
   [0. 0. 0.]
   [0. 0. 0.]]

  [[0. 0. 0.]
   [0. 0. 0

array([[[[-10.44078,   0.02876,   0.05406],
         [  0.02876, -10.70637,   0.00629],
         [  0.05406,   0.00629, -10.73214]],

        [[ -0.39338,  -0.01632,  -0.06839],
         [ -0.01275,  -0.00935,   0.00201],
         [ -0.05327,   0.00219,  -0.03053]],

        [[ -0.03564,  -0.01293,   0.00348],
         [ -0.01668,  -0.15005,  -0.0148 ],
         [  0.0057 ,  -0.00539,  -0.04819]],

        [[ -0.05   ,   0.0036 ,   0.01328],
         [  0.0038 ,  -0.05259,   0.00828],
         [ -0.0039 ,  -0.00122,  -0.0953 ]]],


       [[[ -0.39338,  -0.01275,  -0.05327],
         [ -0.01632,  -0.00935,   0.00219],
         [ -0.06839,   0.00201,  -0.03053]],

        [[  0.43471,  -0.01889,   0.03271],
         [ -0.01889,   0.0347 ,   0.00096],
         [  0.03271,   0.00096,   0.03504]],

        [[ -0.02082,   0.02822,   0.00035],
         [  0.03824,  -0.04083,  -0.00522],
         [ -0.00577,   0.00187,   0.02012]],

        [[ -0.02214,   0.00316,   0.01997],
         [ -0.00

## e1 分解 (hcore + overlap)

`e1` 包含 core Hamiltonian 二阶导数与 overlap 二阶导数两部分：

- **de\_hcore**: $\sum_{A,B} \langle \nabla_A \nabla_B H_{\text{core}} | D_0 \rangle$
- **de\_ovlp**: $-\sum_{A,B} \langle \nabla_A \nabla_B S | D_{E,0} \rangle$ (能量加权密度矩阵)

e1 对所有 `auxbasis_response` 值相同 (不含辅助基响应)。

In [11]:
# e1 is the same for all auxbasis_response levels (no aux dependence)
e1 = e1_aux2.copy()

hcore_deriv = mf_hess.hcore_generator(mol)
s1aa, s1ab, s1a_ovlp = rhf_hess.get_ovlp(mol)

de_hcore = np.zeros((natm, natm, 3, 3))
de_ovlp = np.zeros((natm, natm, 3, 3))

for i0, ia in enumerate(atmlst):
    shl0, shl1, p0, p1 = aoslices[ia]
    # overlap diagonal: s1aa contracted with dme0
    de_ovlp[i0, i0] -= np.einsum('xypq,pq->xy', s1aa[:, :, p0:p1], dme0[p0:p1]) * 2
    for j0, ja in enumerate(atmlst[:i0 + 1]):
        q0, q1 = aoslices[ja][2:]
        # hcore second derivative contracted with dm0
        h1ao_hc = hcore_deriv(ia, ja)
        de_hcore[i0, j0] += np.einsum('xypq,pq->xy', h1ao_hc, dm0)
        # overlap cross: s1ab contracted with dme0
        de_ovlp[i0, j0] -= np.einsum('xypq,pq->xy', s1ab[:, :, p0:p1, q0:q1], dme0[p0:p1, q0:q1]) * 2
    for j0 in range(i0):
        de_hcore[j0, i0] = de_hcore[i0, j0].T
        de_ovlp[j0, i0] = de_ovlp[i0, j0].T

print("e1 = hcore + ovlp:", np.allclose(e1, de_hcore + de_ovlp))

e1 = hcore + ovlp: True


## J 贡献分解

In [12]:
# J_basis_2nd = ej with auxbasis_response = 0 (orbital-only derivatives)
# This includes (20|0)(0|00), (11|0)(0|00), (10|0)(0|10) contributions
de_J20 = ej_aux0.copy()

# J_basis_1st_aux_1st: full 1st-order aux response with correct factor 2.0
# aux1 gives factor 1.0, aux2 gives factor 2.0, so we scale the difference by 2
de_J11 = 2.0 * (ej_aux1 - ej_aux0)

# J_aux_2nd: 2nd-order aux response
de_J02 = ej_aux2 - 2.0 * ej_aux1 + ej_aux0

print("J decomposition check:", np.allclose(ej_aux2, de_J20 + de_J11 + de_J02))

J decomposition check: True


## K 贡献分解

In [13]:
# K_basis_2nd = ek with auxbasis_response = 0 (orbital-only derivatives)
de_K20 = ek_aux0.copy()

# K_basis_1st_aux_1st: full 1st-order aux response with correct factor 1.0
# aux1 gives factor 0.5, aux2 gives factor 1.0, so we scale the difference by 2
de_K11 = 2.0 * (ek_aux1 - ek_aux0)

# K_aux_2nd: 2nd-order aux response
de_K02 = ek_aux2 - 2.0 * ek_aux1 + ek_aux0

print("K decomposition check:", np.allclose(ek_aux2, de_K20 + de_K11 + de_K02))

K decomposition check: True


## CPHF 响应

In [14]:
# Compute full hess_elec = partial_hess_elec + CPHF response
de_hess_elec = mf_hess.hess_elec()

# CPHF response = hess_elec - partial_hess_elec
de_partial = e1 + ej_aux2 - ek_aux2
de_cphf = de_hess_elec - de_partial

print("partial_hess_elec check:", np.allclose(de_partial, mf_hess.partial_hess_elec()))
print("hess_elec = partial + cphf:", np.allclose(de_hess_elec, de_partial + de_cphf))

tmp_contrib1, fp: -40.4820033252071
tmp_contrib2, fp: 40.49762230517417
tmp_contrib3, fp: -4.423643843762165
tmp_contrib4, fp: -20362.502758972907
tmp_contrib5, fp: 10181.02765040996
tmp_contrib6, fp: 10181.475120771258
tmp_contrib7, fp: 100.26786686092383
tmp_contrib8, fp: -100.33444387990247
tmp_contrib1, fp: -40.4820033252071
[[[[-218.0347     0.01337    0.57194]
   [   0.01337 -219.39356    0.244  ]
   [   0.57194    0.244   -219.68089]]

  [[   0.         0.         0.     ]
   [   0.         0.         0.     ]
   [   0.         0.         0.     ]]

  [[   0.         0.         0.     ]
   [   0.         0.         0.     ]
   [   0.         0.         0.     ]]

  [[   0.         0.         0.     ]
   [   0.         0.         0.     ]
   [   0.         0.         0.     ]]]


 [[[   0.         0.         0.     ]
   [   0.         0.         0.     ]
   [   0.         0.         0.     ]]

  [[   0.0943     0.06305    0.18068]
   [   0.06305   -1.07026    0.02518]
   [   0.18

## 总核验

In [15]:
de_sum = de_hcore + de_ovlp \
         + de_J20 + de_J11 + de_J02 \
         - de_K20 - de_K11 - de_K02 \
         + de_cphf + de_nuc

print("de_ref == de_sum:", np.allclose(de_ref, de_sum))
print("max abs difference:", np.max(np.abs(de_ref - de_sum)))

de_ref == de_sum: True
max abs difference: 5.686562332130052e-13


In [16]:
print("========== Contribution Summary ==========")
contributions = {
    "hcore":               de_hcore,
    "ovlp":                de_ovlp,
    "J_basis_2nd":         de_J20,
    "J_basis_1st_aux_1st": de_J11,
    "J_aux_2nd":           de_J02,
    "K_basis_2nd":         de_K20,
    "K_basis_1st_aux_1st": de_K11,
    "K_aux_2nd":           de_K02,
    "cphf":                de_cphf,
    "nuc":                 de_nuc,
}

for name, arr in contributions.items():
    sign = "+" if name.startswith("J") or name in ["hcore", "ovlp", "cphf", "nuc"] else "-"
    print(f"  {sign} {name:30s}: max = {np.max(np.abs(arr)):12.6f}, norm = {np.linalg.norm(arr):12.6f}")

print(f"\n  {'de_ref':30s}: max = {np.max(np.abs(de_ref)):12.6f}, norm = {np.linalg.norm(de_ref):12.6f}")

========== Contribution Summary ==========
  + hcore                         : max =     4.796595, norm =    15.424534
  + ovlp                          : max =     0.471350, norm =     1.219724
  + J_basis_2nd                   : max =    26.681874, norm =    46.496180
  + J_basis_1st_aux_1st           : max =    44.752165, norm =    77.479630
  + J_aux_2nd                     : max =    22.376444, norm =    38.740809
  - K_basis_2nd                   : max =    11.977857, norm =    20.655346
  - K_basis_1st_aux_1st           : max =    22.918044, norm =    39.671048
  - K_aux_2nd                     : max =    11.459257, norm =    19.835893
  + cphf                          : max =     0.257821, norm =     0.743761
  + nuc                           : max =     1.810204, norm =     5.912849

  de_ref                        : max =     0.478794, norm =     1.008162


# 详细分解

详细分解的大体原则：

- 程序越简单越好。完全不考虑效率。
- 我们当前不考虑所有电子积分的对称性。所有电子积分直接存到对应的变量里。
- 所有运算使用 einsum (如果要在外部 Python 程序执行，记得要加 optimize=True，即使效率在这里不关键但不加该选项的执行时间会非常大；本 notebook 是因为前面有了 `functools.partial` 重定义了 np.einsum 的默认参数为 `optimize="greedy"`，所以可以不用再加)。
- 首先得到一个关于基组 / 辅助基的、与原子无关的贡献矩阵 (例如 `dbas_J_20_ip1_contrib`)；这个矩阵应该需要足够小 (一般是 3 x 3 x nao/naux x nao/naux)，这个储存大小比较小，方便后续处理。
- 然后通过一个双重循环 (A, B) 来将基组 / 辅助基的贡献矩阵转换为原子贡献矩阵 (例如 `de_J_20_ip1_contrib[A, B]`)。留意有一些贡献项是只对单个原子 (A) 循环的 (例如 `de_J_20_ipip1_contrib[A, A]`)。这里也同时处理一些系数缩放 (譬如 `de_J20_ip1_contrib` 的 4 倍)。
- 最后将结果拼起来，用 `np.allclose` 检查是否相等。

- 对于 K (交换积分) 的贡献，我们经常需要使用 `occ_coeff` 与 `occ_occupation` 变量以对其中一个原子轨道，缩并到分子轨道。

### J (basis_2nd)

In [17]:
int2c2e = aux.intor("int2c2e")
int2c2e_inv = np.linalg.inv(int2c2e)
int3c2e = _int3c_wrapper(mol, aux, "int3c2e", "s1")()
int3c2e_ip1 = _int3c_wrapper(mol, aux, "int3c2e_ip1", "s1")().reshape([3, nao, nao, naux])
int3c2e_ip2 = _int3c_wrapper(mol, aux, "int3c2e_ip2", "s1")().reshape([3, nao, nao, naux])
int3c2e_ipip1 = _int3c_wrapper(mol, aux, "int3c2e_ipip1", "s1")().reshape([3, 3, nao, nao, naux])
int3c2e_ipvip1 = _int3c_wrapper(mol, aux, "int3c2e_ipvip1", "s1")().reshape([3, 3, nao, nao, naux])
int3c2e_ip1ip2 = _int3c_wrapper(mol, aux, "int3c2e_ip1ip2", "s1")().reshape([3, 3, nao, nao, naux])
int3c2e_ipip2 = _int3c_wrapper(mol, aux, "int3c2e_ipip2", "s1")().reshape([3, 3, nao, nao, naux])
int2c2e_ip1 = aux.intor("int2c2e_ip1")
int2c2e_ipip1 = aux.intor("int2c2e_ipip1").reshape([3, 3, naux, naux])
int2c2e_ip1ip2 = aux.intor("int2c2e_ip1ip2").reshape([3, 3, naux, naux])


In [18]:
# (10|0)(0|10)
dbas_J02_contrib1 = np.einsum("tuvP, PQ, sklQ, uv, kl -> tsuk", int3c2e_ip1, int2c2e_inv, int3c2e_ip1, dm0, dm0)
de_J02_contrib1 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(aoslices):
    for B, (_, _, p0B, p1B) in enumerate(aoslices):
        de_J02_contrib1[A, B] += 4 * np.einsum("tsuv -> ts", dbas_J02_contrib1[:, :, p0A:p1A, p0B:p1B])

In [19]:
# (11|0)(0|00)
dbas_J02_contrib2 = np.einsum("tsuvP, PQ, klQ, kl -> tsuv", int3c2e_ipvip1, int2c2e_inv, int3c2e, dm0)
de_J02_contrib2 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(aoslices):
    for B, (_, _, p0B, p1B) in enumerate(aoslices):
        de_J02_contrib2[A, B] += 2 * np.einsum("tsuv, uv -> ts", dbas_J02_contrib2[:, :, p0A:p1A, p0B:p1B], dm0[p0A:p1A, p0B:p1B])

In [20]:
# (20|0)(0|00)
dbas_J20_contrib3 = np.einsum("tsuvP, PQ, klQ, kl -> tsuv", int3c2e_ipip1, int2c2e_inv, int3c2e, dm0)
de_J20_contrib3 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(aoslices):
    de_J20_contrib3[A, A] += 2 * np.einsum("tsuv, uv -> ts", dbas_J20_contrib3[:, :, p0A:p1A], dm0[p0A:p1A])

In [21]:
de_J20_recap = de_J02_contrib1 + de_J02_contrib2 + de_J20_contrib3
np.allclose(de_J20_recap, de_J20)

True

### J (basis_1st ux_1st)

In [22]:
# (10|1)(0|0)(0|00)
dbas_J11_contrib1 = np.einsum("tsuvP, PQ, klQ, uv, kl -> tsuP", int3c2e_ip1ip2, int2c2e_inv, int3c2e, dm0, dm0)
de_J11_contrib1 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(aoslices):
    for B, (_, _, p0B, p1B) in enumerate(auxslices):
        de_J11_contrib1[A, B] += 2 * np.einsum("tsuP -> ts", dbas_J11_contrib1[:, :, p0A:p1A, p0B:p1B])
de_J11_contrib1 += de_J11_contrib1.transpose(1, 0, 3, 2)

In [23]:
# (10|0)(0|1)(0|00)
dbas_J11_contrib2 = np.einsum("tuvP, PQ, sQR, RS, klS, uv, kl -> tsuR", int3c2e_ip1, int2c2e_inv, int2c2e_ip1, int2c2e_inv, int3c2e, dm0, dm0)
de_J11_contrib2 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(aoslices):
    for B, (_, _, p0B, p1B) in enumerate(auxslices):
        de_J11_contrib2[A, B] += 2 * np.einsum("tsuR -> ts", dbas_J11_contrib2[:, :, p0A:p1A, p0B:p1B])
de_J11_contrib2 += de_J11_contrib2.transpose(1, 0, 3, 2)

In [24]:
# (10|0)(1|0)(0|00)
dbas_J11_contrib3 = np.einsum("tuvP, PQ, sQR, RS, klS, uv, kl -> tsuQ", int3c2e_ip1, int2c2e_inv, int2c2e_ip1, int2c2e_inv, int3c2e, dm0, dm0)
de_J11_contrib3 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(aoslices):
    for B, (_, _, p0B, p1B) in enumerate(auxslices):
        de_J11_contrib3[A, B] += -2 * np.einsum("tsuQ -> ts", dbas_J11_contrib3[:, :, p0A:p1A, p0B:p1B])
de_J11_contrib3 += de_J11_contrib3.transpose(1, 0, 3, 2)

In [25]:
# (10|0)(0|0)(1|00)
dbas_J11_contrib4 = np.einsum("tuvP, PQ, sklQ, uv, kl -> tsuQ", int3c2e_ip1, int2c2e_inv, int3c2e_ip2, dm0, dm0)
de_J11_contrib4 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(aoslices):
    for B, (_, _, p0B, p1B) in enumerate(auxslices):
        de_J11_contrib4[A, B] += 2 * np.einsum("tsuQ -> ts", dbas_J11_contrib4[:, :, p0A:p1A, p0B:p1B])
de_J11_contrib4 += de_J11_contrib4.transpose(1, 0, 3, 2)

In [26]:
de_J11_recap = de_J11_contrib1 + de_J11_contrib2 + de_J11_contrib3 + de_J11_contrib4
np.allclose(de_J11_recap, de_J11)

True

### J (aux_2nd)

In [27]:
# (00|2)(0|00)
dbas_J02_contrib1 = np.einsum("tsuvP, PQ, klQ, uv, kl -> tsP", int3c2e_ipip2, int2c2e_inv, int3c2e, dm0, dm0)
de_J02_contrib1 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(auxslices):
    de_J02_contrib1[A, A] += np.einsum("tsP -> ts", dbas_J02_contrib1[:, :, p0A:p1A])

In [28]:
# (00|0)(2|0)(0|00)
dbas_J02_contrib2 = np.einsum("uvP, PQ, tsQR, RS, klS, uv, kl -> tsQ", int3c2e, int2c2e_inv, int2c2e_ipip1, int2c2e_inv, int3c2e, dm0, dm0)
de_J02_contrib2 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(auxslices):
    de_J02_contrib2[A, A] += -1 * np.einsum("tsQ -> ts", dbas_J02_contrib2[:, :, p0A:p1A])
de_J02_contrib2 = de_J02_contrib2

In [29]:
# (00|0)(1|1)(0|00)
dbas_J02_contrib3a = np.einsum("uvP, PQ, tsQR, RS, klS, uv, kl -> tsQR", int3c2e, int2c2e_inv, int2c2e_ip1ip2, int2c2e_inv, int3c2e, dm0, dm0)
de_J02_contrib3a = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(auxslices):
    for B, (_, _, p0B, p1B) in enumerate(auxslices):
        de_J02_contrib3a[A, B] += -0.5 * np.einsum("tsQR -> ts", dbas_J02_contrib3a[:, :, p0A:p1A, p0B:p1B])
de_J02_contrib3a += de_J02_contrib3a.transpose(1, 0, 3, 2)

In [30]:
# (00|0)(1|0)(0|1)(0|00)
dbas_J02_contrib3b = np.einsum("uvP, PQ, tQR, RS, sST, TU, klU, uv, kl -> tsQT", int3c2e, int2c2e_inv, int2c2e_ip1, int2c2e_inv, int2c2e_ip1, int2c2e_inv, int3c2e, dm0, dm0)
de_J02_contrib3b = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(auxslices):
    for B, (_, _, p0B, p1B) in enumerate(auxslices):
        de_J02_contrib3b[A, B] += -0.5 * np.einsum("tsQT -> ts", dbas_J02_contrib3b[:, :, p0A:p1A, p0B:p1B])
de_J02_contrib3b += de_J02_contrib3b.transpose(1, 0, 3, 2)

In [31]:
# (00|1)(1|0)(0|00)
dbas_J02_contrib4 = np.einsum("tuvP, PQ, sQR, RS, klS, uv, kl -> tsPQ", int3c2e_ip2, int2c2e_inv, int2c2e_ip1, int2c2e_inv, int3c2e, dm0, dm0)
de_J02_contrib4 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(auxslices):
    for B, (_, _, p0B, p1B) in enumerate(auxslices):
        de_J02_contrib4[A, B] += -1 * np.einsum("tsPQ -> ts", dbas_J02_contrib4[:, :, p0A:p1A, p0B:p1B])
de_J02_contrib4 += de_J02_contrib4.transpose(1, 0, 3, 2)

In [32]:
# (00|1)(1|00)
dbas_J02_contrib5 = np.einsum("tuvP, PQ, sklQ, uv, kl -> tsPQ", int3c2e_ip2, int2c2e_inv, int3c2e_ip2, dm0, dm0)
de_J02_contrib5 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(auxslices):
    for B, (_, _, p0B, p1B) in enumerate(auxslices):
        de_J02_contrib5[A, B] += 0.5 * np.einsum("tsPQ -> ts", dbas_J02_contrib5[:, :, p0A:p1A, p0B:p1B])
de_J02_contrib5 += de_J02_contrib5.transpose(1, 0, 3, 2)

In [33]:
# (00|0)(0|1)(1|0)(0|00)
dbas_J02_contrib6 = np.einsum("uvP, PQ, tRQ, RS, sST, TU, klU, uv, kl -> tsRS", int3c2e, int2c2e_inv, int2c2e_ip1, int2c2e_inv, int2c2e_ip1, int2c2e_inv, int3c2e, dm0, dm0)
de_J02_contrib6 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(auxslices):
    for B, (_, _, p0B, p1B) in enumerate(auxslices):
        de_J02_contrib6[A, B] += 0.5 * np.einsum("tsRS -> ts", dbas_J02_contrib6[:, :, p0A:p1A, p0B:p1B])
de_J02_contrib6 += de_J02_contrib6.transpose(1, 0, 3, 2)

In [34]:
# (00|1)(0|1)(0|00)
dbas_J02_contrib7 = np.einsum("tuvP, PQ, sRQ, RS, klS, uv, kl -> tsPR", int3c2e_ip2, int2c2e_inv, int2c2e_ip1, int2c2e_inv, int3c2e, dm0, dm0)
de_J02_contrib7 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(auxslices):
    for B, (_, _, p0B, p1B) in enumerate(auxslices):
        de_J02_contrib7[A, B] += -1 * np.einsum("tsPR -> ts", dbas_J02_contrib7[:, :, p0A:p1A, p0B:p1B])
de_J02_contrib7 += de_J02_contrib7.transpose(1, 0, 3, 2)

In [35]:
# (00|0)(1|0)(1|0)(0|00)
dbas_J02_contrib8 = np.einsum("uvP, PQ, tQR, RS, sST, TU, klU, uv, kl -> tsRT", int3c2e, int2c2e_inv, int2c2e_ip1, int2c2e_inv, int2c2e_ip1, int2c2e_inv, int3c2e, dm0, dm0)
de_J02_contrib8 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(auxslices):
    for B, (_, _, p0B, p1B) in enumerate(auxslices):
        de_J02_contrib8[A, B] += 1 * np.einsum("tsRT -> ts", dbas_J02_contrib8[:, :, p0A:p1A, p0B:p1B])
de_J02_contrib8 += de_J02_contrib8.transpose(1, 0, 3, 2)

In [36]:
de_J02_recap = de_J02_contrib1 + de_J02_contrib2 + de_J02_contrib3a + de_J02_contrib3b + de_J02_contrib4 + de_J02_contrib5 + de_J02_contrib6 + de_J02_contrib7 + de_J02_contrib8
np.allclose(de_J02_recap, de_J02, atol=1e-5, rtol=1e-4)

True